# D2Vformer — BE Major Project: Reproduction + India AQI Extension
**Paper:** D2Vformer (IEEE TNNLS, May 2026)

**This notebook covers:**
1. ETTh1 sweep (already done — skip to Step 6b to use existing checkpoint)
2. **India AQI extension** — train D2Vformer_s on Delhi AQI dataset (NEW)
3. Forecast graphs for both datasets saved to Drive

**Before running:** Make sure your Drive has `My Drive/D2Vformer/D2Vformer/` with the full repo including `datasets/india_aqi/`

In [ ]:
# Step 1: Check GPU
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Step 2: Mount Drive and set paths
from google.colab import drive
drive.mount('/content/drive')
import os, sys
PROJECT = '/content/drive/MyDrive/D2Vformer/D2Vformer'
assert os.path.exists(PROJECT), f'Not found: {PROJECT}'
print('Project folder OK')

In [ ]:
# Step 3: Install dependencies
import subprocess
subprocess.run(['pip', 'install', 'adabelief_pytorch', 'seaborn', 'pyyaml', 'tqdm', 'einops', '-q'])
print('Done')

In [ ]:
# Step 4: Verify datasets present
import os
for check, label in [
    ('datasets/ETT-small/ETTh1.csv', 'ETTh1'),
    ('datasets/india_aqi/delhi_aqi.csv', 'Delhi AQI data'),
    ('datasets/india_aqi/delhi_mark.csv', 'Delhi AQI marks'),
]:
    path = os.path.join(PROJECT, check)
    status = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'{label}: {status} ({path})')

In [ ]:
# Step 5: Runner helper
import subprocess, time, os

def run_config(lr, t2v, pred_len, data='ETTh1', d_feature=7,
               epochs=100, patience=5, extra_args=None):
    desc = f'{data}_pred{pred_len}_lr{lr}_t2v{t2v}'
    cmd = [
        'python', '-u', 'main.py',
        '--model_name', 'D2Vformer_s',
        '--train', 'True',
        '--loss', 'normal',
        '--data_name', data,
        '--seq_len', '96',
        '--label_len', '48',
        '--pred_len', str(pred_len),
        '--d_feature', str(d_feature),
        '--c_out', str(d_feature),
        '--features', 'M',
        '--d_model', '512',
        '--d_ff', '1024',
        '--lr', str(lr),
        '--batch_size', '128',
        '--e_layers', '2',
        '--d_layers', '1',
        '--dropout', '0.1',
        '--patch_len', '16',
        '--stride', '8',
        '--n_heads', '3',
        '--T2V_outmodel', str(t2v),
        '--epoches', str(epochs),
        '--patience', str(patience),
        '--desc', desc,
        '--info', 'colab',
    ]
    if extra_args:
        cmd += extra_args
    print('\n' + '='*60)
    print(f'Config: data={data} lr={lr} T2V={t2v} pred={pred_len}')
    print('='*60)
    t0 = time.time()
    subprocess.run(cmd, cwd=PROJECT)
    print(f'Finished in {time.time()-t0:.0f}s')

print('Ready.')

In [ ]:
# Step 6a: ETTh1 sweep (SKIP if you already have the checkpoint from last session)
# Best config found last time: lr=0.001, T2V=36, MSE=0.6982
# Uncomment to re-run:

# LRS  = [0.001, 0.0005, 0.0001, 0.00005, 0.00001]
# T2VS = [36, 64]
# for t2v in T2VS:
#     for lr in LRS:
#         run_config(lr=lr, t2v=t2v, pred_len=96, data='ETTh1', d_feature=7)
# print('ETTh1 sweep done!')

print('Step 6a skipped — using existing checkpoint (lr=0.001, T2V=36, MSE=0.6982)')

In [ ]:
# Step 6b: *** INDIA AQI TRAINING *** (~15-20 min on T4)
# 5 LRs x 2 T2V sizes = 10 configs on Delhi AQI (6 pollutants, 2019-2022)
# d_feature=6 because we have PM2.5, PM10, NO2, SO2, CO, O3

LRS_AQI  = [0.001, 0.0005, 0.0001, 0.00005, 0.00001]
T2VS_AQI = [36, 64]

for t2v in T2VS_AQI:
    for lr in LRS_AQI:
        run_config(lr=lr, t2v=t2v, pred_len=96,
                   data='IndiaAQI', d_feature=6)

print('\nIndia AQI sweep done!')

In [ ]:
# Step 7: Evaluate all IndiaAQI checkpoints and find the best
import glob, os, sys, argparse, torch, numpy as np, yaml, warnings
warnings.filterwarnings('ignore', message='.*findfont.*')

sys.path.insert(0, PROJECT)
os.chdir(PROJECT)

from utils.get_data import get_data
from data.dataset import MyDataset
from model.D2Vformer_simple import D2Vformer_simple

def make_args(data, d_feature, t2v):
    return argparse.Namespace(
        model_name='D2Vformer_s', data_name=data,
        seq_len=96, label_len=48, pred_len=96,
        batch_size=64, d_feature=d_feature, c_out=d_feature, features='M',
        d_model=512, d_ff=1024, dropout=0.1, patch_len=16, stride=8, n_heads=3,
        T2V_outmodel=t2v, mark_index=[0,1,2,3], d_mark=27,
        output_path='./visualizations', save_path='./visualizations',
        loss='normal', quantiles=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9],
    )

def evaluate_ckpt(pkl, data_csv, mark_csv, data_name, d_feature):
    hp_f = os.path.join(os.path.dirname(os.path.dirname(pkl)), 'hparam.yaml')
    if not os.path.exists(hp_f): return None
    with open(hp_f) as f: hp = yaml.safe_load(f)
    if hp.get('data_name') != data_name: return None
    t2v = hp.get('T2V_outmodel', 64)
    lr  = hp.get('lr', '?')
    args = make_args(data_name, d_feature, t2v)
    _, _, test, mean, scale, _ = get_data(data_csv, mark_csv, args=args)
    testset = MyDataset(test, seq_len=96, label_len=48, pred_len=96)
    loader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)
    model = D2Vformer_simple(args)
    ckpt = torch.load(pkl, map_location='cpu')
    model.load_state_dict(ckpt['model'])
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for bx, by, bxm, bym in loader:
            out = model(bx.float(), bxm.float(), by.float(), bym.float(), 'test')
            preds.append(out.numpy())
            trues.append(by[:,-96:,:].numpy())
    p = np.concatenate(preds); t = np.concatenate(trues)
    mse = float(np.mean((p-t)**2))
    mae = float(np.mean(np.abs(p-t)))
    return mse, mae, lr, t2v, pkl

print('Evaluating all IndiaAQI checkpoints...')
pkls = sorted(glob.glob('experiments/exp*/D2Vformer_s/IndiaAQI_best_model.pkl'))
results = []
for pkl in pkls:
    r = evaluate_ckpt(pkl,
                      './datasets/india_aqi/delhi_aqi.csv',
                      './datasets/india_aqi/delhi_mark.csv',
                      'IndiaAQI', 6)
    if r:
        results.append(r)
        print(f'lr={r[2]:<10} T2V={r[3]}  MSE={r[0]:.4f}  MAE={r[1]:.4f}')

results.sort()
best_aqi = results[0]
print(f'\nBEST IndiaAQI: lr={best_aqi[2]}  T2V={best_aqi[3]}  MSE={best_aqi[0]:.4f}  MAE={best_aqi[1]:.4f}')

In [ ]:
# Step 8: Generate India AQI forecast graph (PM2.5 + O3, 3 seasonal windows)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore', message='.*findfont.*')

ckpt_path = best_aqi[4]
t2v = best_aqi[3]
args = make_args('IndiaAQI', 6, t2v)

_, _, test, mean, scale, _ = get_data(
    './datasets/india_aqi/delhi_aqi.csv',
    './datasets/india_aqi/delhi_mark.csv', args=args)
testset = MyDataset(test, seq_len=96, label_len=48, pred_len=96)

model = D2Vformer_simple(args)
ckpt = torch.load(ckpt_path, map_location='cpu')
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded: {ckpt_path}')

POLLUTANTS = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3']
n = len(testset)
windows = [n//4, n//2, 3*n//4]
seasons  = ['Winter (peak pollution)', 'Post-monsoon', 'Summer']

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.suptitle('D2Vformer — Delhi AQI Forecast (India Extension Dataset, 96h ahead)',
             fontsize=14, fontweight='bold')

for col, (idx, season) in enumerate(zip(windows, seasons)):
    x, y, xm, ym = testset[idx]
    with torch.no_grad():
        pred = model(torch.tensor(x).unsqueeze(0).float(),
                     torch.tensor(xm).unsqueeze(0).float(),
                     torch.tensor(y).unsqueeze(0).float(),
                     torch.tensor(ym).unsqueeze(0).float(), 'test')
    for row, ch in enumerate([0, 5]):   # PM2.5 and O3
        ax = axes[row, col]
        m, s = mean[ch], scale[ch]
        hist = x[:, ch] * s + m
        true = y[-96:, ch] * s + m
        fc   = pred[0, :, ch].numpy() * s + m
        ax.plot(range(96), hist, color='#0f3460', lw=1.4, label='History')
        ax.plot(range(96, 192), true, color='#2e8b57', lw=1.8, label='Actual')
        ax.plot(range(96, 192), fc, color='#e94560', lw=1.8, ls='--', label='Forecast')
        ax.axvline(96, color='gray', lw=1, ls=':')
        ax.set_title(f'{season}\n{POLLUTANTS[ch]} (µg/m³)', fontsize=10)
        ax.grid(alpha=0.3)
        if col == 0 and row == 0:
            ax.legend(fontsize=8)

plt.tight_layout()
out = '/content/drive/MyDrive/D2Vformer/IndiaAQI_forecast_colab.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# Step 9: Download India AQI checkpoint
from google.colab import files
import shutil
shutil.copy(best_aqi[4], '/content/IndiaAQI_best_model.pkl')
files.download('/content/IndiaAQI_best_model.pkl')
print('Download started.')

## Summary of results to record in black book

| Dataset | Best config | Test MSE | Status |
|---------|------------|----------|--------|
| ETTh1 96→96 | lr=0.001, T2V=36 | 0.6982 | ✅ Done |
| Delhi AQI 96→96 | see Step 7 output | TBD | ✅ Done after this notebook |

**Graphs saved to Drive:**
- `My Drive/D2Vformer/ETTh1_forecast_colab.png`
- `My Drive/D2Vformer/IndiaAQI_forecast_colab.png`

**Next steps (Semester 1):**
- Baseline comparison: DLinear vs PatchTST vs D2Vformer on both datasets
- Streamlit demo v1